# **Лабораторная работа** №5

## **Тема: Приложение для анализа заказов интернет-магазина**

## **Цель работы**

Разработать приложение, выполняющее анализ заказов и формирующее персонализированный отчёт с использованием различных библиотек Python.






## **Задание**

Создать программу, работающую по следующему сценарию:


### **1. Ввод и обработка данных пользователя**

Запросить дату рождения пользователя (формат YYYY-MM-DD).

Использовать:

* `datetime` — для получения текущей даты и вычисления возраста в днях;
* `calendar` — для определения дня недели рождения;
* `re` — для проверки корректности формата ввода.

Результат:

* возраст в днях;
* день недели (в текстовом виде).

👉 Эти данные используются в итоговом отчёте.


### **2. Загрузка и подготовка данных заказов**

Загрузить файл `orders.csv`.

Использовать:

* `pandas` — для загрузки данных (`read_csv`), обработки таблицы и преобразования столбца с датой (`to_datetime`).

Выполнить:

* проверку типов данных;
* обработку пропущенных значений;
* удаление некорректных записей.

👉 Подготовленные данные используются далее для анализа.


### **3. Расчёт показателей**

Добавить новые столбцы:

* общая стоимость заказа;
* итоговая стоимость с учётом скидки;
* признак успешной продажи.

Использовать:

* `pandas` — для вычислений по столбцам DataFrame.

👉 Эти значения являются основой для аналитики.


### **4. Анализ данных**

Выполнить расчёты:

* общий доход;
* доход по категориям;
* топ товаров;
* уровень возвратов.

Использовать:

* `pandas` — группировка (`groupby`), агрегации (`sum`, `mean`, `count`).

👉 Результаты включаются в итоговый отчёт.


### **5. Мат расчёт**

Реализовать вычисление, например:

* оценка времени доставки на основе расстояния.
* ... (по 3 индивидуальных расчета на ваше усмотрение)

Использовать:

* `math` — математические функции (например, `sqrt`);
* `math.isnan()` — проверка корректности числового ввода.

👉 Результат включается в отчёт.


### **6. Формирование отчёта**

Сформировать текстовый отчёт, содержащий:

* данные пользователя;
* результаты анализа;
* дополнительные вычисления.

Использовать:

* `pandas` — для сохранения результатов в CSV (`to_csv`);
* стандартные средства Python — для формирования текста отчёта.

👉 Отчёт должен быть сохранён в файл.


### **7. Озвучивание отчёта**

Преобразовать текст отчёта в аудио (.mp3).

Использовать:

* `gTTS` — генерация речи из текста.

Проверить:

* что текст не пустой;
* корректность выбранного языка.

### **8. Интерфейс пользователя**

Создать интерфейс для работы с программой.

Использовать:

* `tkinter` — для создания графического интерфейса (ввод данных, кнопки, вывод результатов).

👉 Интерфейс объединяет все функции приложения.


## **Требования**

* использовать библиотеки: `datetime`, `calendar`, `re`, `math`, `pandas`, `tkinter`, `gTTS`;
* обеспечить обработку ошибок (`try/except`);


## **Результат**

Программа, которая:

* получает данные пользователя;
* анализирует заказы интернет-магазина;
* выполняет вычисления;
* формирует текстовый и аудио-отчёт через интерфейс.


In [2]:
import calendar
from datetime import datetime
import json
import math
import os
import re
import tkinter as tk
from tkinter import messagebox, ttk
import pandas as pd


# ==========================================
# 1. ТЕСТОВЫЕ ДАННЫЕ (ГЕНЕРАЦИЯ)
# ==========================================
def create_sample_csv():
    """Создает тестовый файл orders.csv, если он отсутствует"""
    if not os.path.exists("orders.csv"):
        sample_data = {
            "order_id": [101, 102, 103, 104, 105],
            "order_date": [
                "2026-05-01",
                "2026-05-02",
                "2026-05-03",
                "2026-05-04",
                "2026-05-05",
            ],
            "category": [
                "Электроника",
                "Книги",
                "Электроника",
                "Одежда",
                "Книги",
            ],
            "item_name": [
                "Ноутбук",
                "Руководство Python",
                "Смартфон",
                "Джинсы",
                "Учебник по ИИ",
            ],
            "price": [1200.0, 45.0, 800.0, 60.0, math.nan],  # Проверка на NaN
            "quantity": [1, 2, 1, 3, 1],
            "discount": [0.1, 0.0, 0.15, 0.0, 0.0],
            "status": [
                "Delivered",
                "Delivered",
                "Returned",
                "Delivered",
                "Delivered",
            ],
            "distance_km": [15.5, 4.0, 42.2, 8.5, 12.0],
        }
        df = pd.DataFrame(sample_data)
        df.to_csv("orders.csv", index=False, encoding="utf-8")


# ==========================================
# 2. ЯДРО БИЗНЕС-ЛОГИКИ И АНАЛИТИКИ
# ==========================================
class OrderAnalyzerApp:

    def __init__(self):
        self.user_age_days = 0
        self.user_birth_weekday = ""
        self.report_text = ""

    def process_user_data(self, birth_date_str: str) -> bool:
        """Валидация даты рождения, расчет возраста в днях и дня недели"""
        if not re.match(r"^\d{4}-\d{2}-\d{2}$", birth_date_str):
            raise ValueError("Некорректный формат даты. Используйте YYYY-MM-DD.")

        try:
            birth_date = datetime.strptime(birth_date_str, "%Y-%m-%d")
        except ValueError:
            raise ValueError("Указанная дата не существует.")

        current_date = datetime.now()
        if birth_date > current_date:
            raise ValueError("Дата рождения не может быть в будущем.")

        # Расчет возраста в днях
        delta = current_date - birth_date
        self.user_age_days = delta.days

        # Определение дня недели на русском языке
        day_index = birth_date.weekday()
        weekdays = [
            "Понедельник",
            "Вторник",
            "Среда",
            "Четверг",
            "Пятница",
            "Суббота",
            "Воскресенье",
        ]
        self.user_birth_weekday = weekdays[day_index]
        return True

    def analyze_orders(self, file_path: str) -> str:
        """Загрузка, очистка, расчет показателей и анализ данных из CSV"""
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Файл {file_path} не найден.")

        # Загрузка данных
        df = pd.read_csv(file_path, encoding="utf-8")

        # Преобразование типов и обработка пропущенных значений
        df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
        df["price"] = pd.to_numeric(df["price"], errors="coerce")
        df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
        df["distance_km"] = pd.to_numeric(df["distance_km"], errors="coerce")

        # Удаление некорректных записей с критическими NaN
        df = df.dropna(subset=["order_id", "price", "quantity", "distance_km"])

        # Расчет показателей по столбцам
        df["total_cost"] = df["price"] * df["quantity"]
        df["discounted_cost"] = df["total_cost"] * (1 - df["discount"])
        df["is_successful"] = df["status"] == "Delivered"

        # Аналитические показатели
        total_revenue = df[df["is_successful"]]["discounted_cost"].sum()

        cat_group = (
            df[df["is_successful"]]
            .groupby("category")["discounted_cost"]
            .sum()
        )
        category_revenue_str = "\n".join(
            [f"  - {cat}: {val:.2f} руб." for cat, val in cat_group.items()]
        )

        top_items = (
            df.groupby("item_name")["quantity"].sum().sort_values(ascending=False)
        )
        top_item_name = top_items.index[0] if not top_items.empty else "Нет"

        total_orders = len(df)
        returned_orders = len(df[df["status"] == "Returned"])
        return_rate = (
            (returned_orders / total_orders) * 100 if total_orders > 0 else 0
        )

        # Математические расчеты (модуль math)
        # 1. Расчет времени доставки (t = sqrt(s) * 1.5 + 10 минут)
        df["delivery_time_est"] = df["distance_km"].apply(
            lambda x: math.sqrt(x) * 1.5 + 10 if not math.isnan(x) else 0
        )
        avg_delivery_time = df["delivery_time_est"].mean()

        # 2. Расчет радиуса зоны покрытия
        max_dist = df["distance_km"].max()
        coverage_area = math.pi * math.pow(max_dist, 2) if not math.isnan(max_dist) else 0

        # 3. Среднеквадратичное отклонение объемов заказов
        avg_qty = df["quantity"].mean()
        qty_variance = (
            df["quantity"].apply(lambda x: math.pow(x - avg_qty, 2)).sum()
            / total_orders
        )
        qty_std_dev = math.sqrt(qty_variance)

        # Формирование итогового отчета на русском языке
        self.report_text = (
            f"=== ИТОГОВЫЙ АНАЛИТИЧЕСКИЙ ОТЧЕТ ===\n\n"
            f"[Данные пользователя]\n"
            f"Возраст в днях: {self.user_age_days}\n"
            f"День недели рождения: {self.user_birth_weekday}\n\n"
            f"[Анализ продаж]\n"
            f"Общий чистый доход: {total_revenue:.2f} руб.\n"
            f"Доход по товарным категориям:\n{category_revenue_str}\n"
            f"Самый продаваемый товар (шт): {top_item_name}\n"
            f"Доля возвратов: {return_rate:.2f}%\n\n"
            f"[Математические расчеты]\n"
            f"Среднее время доставки (расчетное): {avg_delivery_time:.1f} мин.\n"
            f"Площадь зоны покрытия логистики: {coverage_area:.2f} кв. км.\n"
            f"Стандартное отклонение объема заказов: {qty_std_dev:.2f}\n"
        )

        # Сохранение результатов анализа в текстовый файл и новый CSV
        with open("final_report.txt", "w", encoding="utf-8") as f:
            f.write(self.report_text)

        df.to_csv("processed_orders.csv", index=False, encoding="utf-8")

        return self.report_text


# ==========================================
# 3. ГРАФИЧЕСКИЙ ИНТЕРФЕЙС (TKINTER)
# ==========================================
class AppInterface:

    def __init__(self, root):
        self.root = root
        self.root.title("Анализ заказов интернет-магазина")
        self.root.geometry("600x500")
        self.analyzer = OrderAnalyzerApp()

        create_sample_csv()  # Генерация исходного файла при старте
        self.create_widgets()

    def create_widgets(self):
        # Панель ввода
        input_frame = ttk.LabelFrame(self.root, text=" Данные пользователя ")
        input_frame.pack(fill="x", padx=10, pady=10)

        ttk.Label(
            input_frame, text="Дата рождения (ГГГГ-ММ-ДД):"
        ).pack(side="left", padx=5, pady=5)
        self.birth_entry = ttk.Entry(input_frame, width=15)
        self.birth_entry.pack(side="left", padx=5, pady=5)
        self.birth_entry.insert(0, "2000-01-01")

        # Кнопка запуска
        btn_frame = ttk.Frame(self.root)
        btn_frame.pack(fill="x", padx=10, pady=5)

        self.btn_run = ttk.Button(
            btn_frame, text="Запустить анализ данных", command=self.run_process
        )
        self.btn_run.pack(side="left", padx=5)

        # Зона отображения текстового отчета
        report_frame = ttk.LabelFrame(self.root, text=" Сформированный отчет ")
        report_frame.pack(fill="both", expand=True, padx=10, pady=10)

        self.text_area = tk.Text(report_frame, wrap="word", font=("Arial", 10))
        self.text_area.pack(fill="both", expand=True, padx=5, pady=5)

    def run_process(self):
        birth_str = self.birth_entry.get().strip()

        try:
            # Расчет по данным пользователя
            self.analyzer.process_user_data(birth_str)

            # Анализ файла таблиц
            report = self.analyzer.analyze_orders("orders.csv")

            # Вывод готового текста в графическое окно
            self.text_area.delete("1.0", tk.END)
            self.text_area.insert(tk.END, report)

            messagebox.showinfo(
                "Успешно",
                "Анализ завершен!\nРезультаты сохранены в файлы final_report.txt и processed_orders.csv",
            )

        except Exception as e:
            messagebox.showerror("Ошибка", str(e))


# ==========================================
# ТОЧКА ВХОДА В ПРОГРАММУ
# ==========================================
if __name__ == "__main__":
    root = tk.Tk()
    app = AppInterface(root)
    root.mainloop()